Maked self attention core idea

👉 “A word can only see past words — not future ones.”

Example:
Sentence: “I love AI”
While predicting “love”, model cannot see “AI”

⚡ Why It Matters
Prevents cheating during training
Enables correct text generation (left → right)
Used in autoregressive models (GPT)

👉 Without masking → model would see future → wrong learning

In [1]:
"""
=============================================================================
  Masked Self-Attention & Masked Multi-Head Attention — Full PyTorch Code
  Transformer Architecture | Part 2
=============================================================================

Components implemented here:
  1.  make_causal_mask()              — lower-triangular causal mask
  2.  make_padding_mask()             — ignore <PAD> tokens
  3.  make_decoder_mask()             — combined causal + padding mask
  4.  ScaledDotProductAttention       — with optional mask parameter
  5.  MaskedMultiHeadAttention        — drop-in replacement for MHA with masking
  6.  DecoderLayer                    — full 3-sub-layer decoder block
  7.  visualize_attention_mask()      — ASCII visualisation helper
  8.  run_demo()                      — end-to-end forward pass

Run:
    python masked_attention.py
"""

import math
import torch
import torch.nn as nn
import torch.nn.functional as F


# =============================================================================
# 1.  MASK UTILITIES
# =============================================================================

def make_causal_mask(seq_len: int, device: torch.device = torch.device("cpu")) -> torch.Tensor:
    """
    Lower-triangular causal (look-ahead) mask.

    Shape : [1, 1, seq_len, seq_len]
    Value :  1  → position is ALLOWED to attend
             0  → position is BLOCKED  (future token)

    Example for seq_len=4:
        tensor([[1, 0, 0, 0],
                [1, 1, 0, 0],
                [1, 1, 1, 0],
                [1, 1, 1, 1]])
    """
    mask = torch.tril(torch.ones(seq_len, seq_len, device=device))
    return mask.unsqueeze(0).unsqueeze(0)           # [1, 1, S, S]


def make_padding_mask(token_ids: torch.Tensor, pad_id: int = 0) -> torch.Tensor:
    """
    Padding mask — 1 for real tokens, 0 for <PAD>.

    Args:
        token_ids : [batch, seq_len]  token index tensor
        pad_id    : integer ID of the padding token (default 0)

    Returns:
        mask : [batch, 1, 1, seq_len]  broadcastable over heads & query dim
    """
    mask = (token_ids != pad_id).float()            # [B, S]
    return mask.unsqueeze(1).unsqueeze(2)           # [B, 1, 1, S]


def make_decoder_mask(tgt_ids: torch.Tensor, pad_id: int = 0) -> torch.Tensor:
    """
    Combined decoder mask = causal mask AND padding mask.

    A position j is visible from position i  iff:
        - j <= i        (not in the future), AND
        - j is not PAD  (real token)

    Returns:
        mask : [batch, 1, seq_len, seq_len]
    """
    B, S = tgt_ids.shape
    causal  = make_causal_mask(S, tgt_ids.device)   # [1,  1, S, S]
    padding = make_padding_mask(tgt_ids, pad_id)     # [B,  1, 1, S]
    return causal * padding                          # [B,  1, S, S]


def make_src_mask(src_ids: torch.Tensor, pad_id: int = 0) -> torch.Tensor:
    """
    Padding-only mask for the encoder output (used in cross-attention).
    No causal masking — the decoder can attend to ALL source positions.

    Returns:
        mask : [batch, 1, 1, src_seq_len]
    """
    return make_padding_mask(src_ids, pad_id)


# =============================================================================
# 2.  SCALED DOT-PRODUCT ATTENTION  (with optional mask)
# =============================================================================

class ScaledDotProductAttention(nn.Module):
    """
    Attention(Q, K, V, mask) = softmax((QK^T / sqrt(d_k)) + mask_fill) · V

    The mask is applied BEFORE softmax by filling masked positions with -1e9.
    After softmax, those positions have weight ≈ 0 and contribute nothing.
    """

    def __init__(self, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

    def forward(
        self,
        Q: torch.Tensor,
        K: torch.Tensor,
        V: torch.Tensor,
        mask: torch.Tensor | None = None,
    ):
        """
        Args:
            Q, K, V : [batch, heads, seq_len, d_k]
            mask    : [batch, 1, seq_len, seq_len]  — broadcastable

        Returns:
            output       : [batch, heads, seq_len, d_k]
            attn_weights : [batch, heads, seq_len, seq_len]
        """
        d_k = Q.size(-1)
        # ── Step 1: raw scores ──────────────────────────────────────────────
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
        # scores: [B, h, S, S]

        # ── Step 2: apply mask ──────────────────────────────────────────────
        if mask is not None:
            # Where mask == 0  →  future / padding positions  →  -infinity
            scores = scores.masked_fill(mask == 0, -1e9)

        # ── Step 3: softmax → attention weights ────────────────────────────
        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # ── Step 4: weighted sum of values ─────────────────────────────────
        output = torch.matmul(attn_weights, V)
        return output, attn_weights


# =============================================================================
# 3.  MASKED MULTI-HEAD ATTENTION
# =============================================================================

class MaskedMultiHeadAttention(nn.Module):
    """
    Multi-Head Attention with optional causal / padding mask.

    Used as:
      • Sub-layer 1 in DecoderLayer — with combined causal+padding mask
      • Sub-layer 2 in DecoderLayer — with source padding mask only

    Hyperparameters (original Transformer):
      d_model   = 512
      num_heads = 8
      d_k       = d_v = d_model / num_heads = 64
    """

    def __init__(self, d_model: int = 512, num_heads: int = 8, dropout: float = 0.1):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.d_model   = d_model
        self.num_heads = num_heads
        self.d_k       = d_model // num_heads   # 64

        # Learned projection matrices (no bias — common in practice)
        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)

        self.attention = ScaledDotProductAttention(dropout)

        # Stored after forward() for inspection / visualisation
        self.attn_weights: torch.Tensor | None = None

    # ── internal helper ────────────────────────────────────────────────────
    def _split_heads(self, x: torch.Tensor) -> torch.Tensor:
        """[B, S, d_model]  →  [B, num_heads, S, d_k]"""
        B, S, _ = x.shape
        return (
            x.view(B, S, self.num_heads, self.d_k)
             .transpose(1, 2)
        )

    def _merge_heads(self, x: torch.Tensor) -> torch.Tensor:
        """[B, num_heads, S, d_k]  →  [B, S, d_model]"""
        B, _, S, _ = x.shape
        return (
            x.transpose(1, 2)
             .contiguous()
             .view(B, S, self.d_model)
        )

    # ── forward ───────────────────────────────────────────────────────────
    def forward(
        self,
        query: torch.Tensor,
        key: torch.Tensor,
        value: torch.Tensor,
        mask: torch.Tensor | None = None,
    ) -> torch.Tensor:
        """
        Args:
            query, key, value : [B, S, d_model]
            mask              : [B, 1, S, S]   (optional)

        For self-attention: query == key == value  (same input tensor)
        For cross-attention: query from decoder, key/value from encoder

        Returns:
            output : [B, S, d_model]
        """
        Q = self._split_heads(self.W_Q(query))   # [B, h, S, d_k]
        K = self._split_heads(self.W_K(key))
        V = self._split_heads(self.W_V(value))

        # Masked attention — mask broadcasts over all h heads automatically
        x, self.attn_weights = self.attention(Q, K, V, mask)

        return self.W_O(self._merge_heads(x))    # [B, S, d_model]


# =============================================================================
# 4.  POSITION-WISE FEED-FORWARD NETWORK
# =============================================================================

class PositionwiseFeedForward(nn.Module):
    """FFN(x) = ReLU(x·W1 + b1)·W2 + b2   —   d_model → d_ff → d_model"""

    def __init__(self, d_model: int = 512, d_ff: int = 2048, dropout: float = 0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


# =============================================================================
# 5.  DECODER LAYER
# =============================================================================

class DecoderLayer(nn.Module):
    """
    One Transformer Decoder block (3 sub-layers, each wrapped with Add & Norm):

        1)  Masked Multi-Head Self-Attention   ← uses causal + padding mask
        2)  Cross-Attention (Encoder-Decoder)  ← uses source padding mask only
        3)  Position-wise Feed-Forward Network ← no mask

    Data flow for a single position i in the target:
        tgt → [Masked Self-Attn] → add+norm → [Cross-Attn with enc] → add+norm
            → [FFN] → add+norm → contextualised representation
    """

    def __init__(
        self,
        d_model:   int   = 512,
        num_heads: int   = 8,
        d_ff:      int   = 2048,
        dropout:   float = 0.1,
    ):
        super().__init__()
        self.masked_attn = MaskedMultiHeadAttention(d_model, num_heads, dropout)
        self.cross_attn  = MaskedMultiHeadAttention(d_model, num_heads, dropout)
        self.ffn         = PositionwiseFeedForward(d_model, d_ff, dropout)

        self.norm1   = nn.LayerNorm(d_model)
        self.norm2   = nn.LayerNorm(d_model)
        self.norm3   = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(
        self,
        tgt:      torch.Tensor,
        enc_out:  torch.Tensor,
        tgt_mask: torch.Tensor | None = None,
        src_mask: torch.Tensor | None = None,
    ) -> torch.Tensor:
        """
        Args:
            tgt      : [B, T, d_model]  target token representations
            enc_out  : [B, S, d_model]  encoder output (source context)
            tgt_mask : [B, 1, T, T]     combined causal + padding mask
            src_mask : [B, 1, 1, S]     source padding mask

        Returns:
            tgt : [B, T, d_model]
        """
        # ── Sub-layer 1: Masked Self-Attention ─────────────────────────────
        # Q = K = V = tgt (self-attention).  tgt_mask prevents future peeking.
        _tgt = self.masked_attn(tgt, tgt, tgt, mask=tgt_mask)
        tgt  = self.norm1(tgt + self.dropout(_tgt))

        # ── Sub-layer 2: Cross-Attention ────────────────────────────────────
        # Q = tgt (decoder),  K = V = enc_out (encoder).  No causal mask here.
        _tgt = self.cross_attn(tgt, enc_out, enc_out, mask=src_mask)
        tgt  = self.norm2(tgt + self.dropout(_tgt))

        # ── Sub-layer 3: Feed-Forward ────────────────────────────────────────
        tgt  = self.norm3(tgt + self.dropout(self.ffn(tgt)))
        return tgt


# =============================================================================
# 6.  VISUALISATION HELPER
# =============================================================================

def visualize_attention_mask(mask: torch.Tensor, tokens: list[str] | None = None) -> None:
    """
    Print a colour-coded ASCII grid of an attention mask.

    Args:
        mask   : [seq_len, seq_len]  (2D slice)
        tokens : optional list of token strings for row/col labels
    """
    S = mask.shape[-1]
    labels = tokens or [f"t{i}" for i in range(S)]
    col_w  = max(len(lb) for lb in labels) + 2

    header = " " * col_w + "".join(lb.center(col_w) for lb in labels)
    print(header)
    print("-" * len(header))

    for i, row_label in enumerate(labels):
        row = row_label.ljust(col_w)
        for j in range(S):
            val = mask[i, j].item()
            sym = " 1 " if val == 1 else " . "   # '.' = masked
            row += sym.center(col_w)
        print(row)
    print()


# =============================================================================
# 7.  END-TO-END DEMO
# =============================================================================

def run_demo():
    torch.manual_seed(0)
    print("=" * 66)
    print("  Masked Self-Attention — End-to-End Demo")
    print("=" * 66)

    B, SRC_LEN, TGT_LEN, D = 2, 10, 8, 512

    # ── Causal mask visualisation ─────────────────────────────────────────
    print("\n[1] Causal mask for TGT_LEN=5  (1=allowed, .=blocked)")
    tokens = ["BOS", "I", "love", "cats", "EOS"]
    causal = make_causal_mask(5).squeeze()
    visualize_attention_mask(causal, tokens)

    # ── Build masks ───────────────────────────────────────────────────────
    print("[2] Building decoder mask (causal + padding)...")
    # Batch item 0 has 2 PAD tokens at the end; item 1 has none
    tgt_ids = torch.tensor([[4, 7, 12,  3,  6,  9,  0,  0],   # 0=PAD
                             [4, 7, 12,  3,  6,  9,  1,  2]])
    src_ids = torch.ones(B, SRC_LEN, dtype=torch.long)  # no src padding

    tgt_mask = make_decoder_mask(tgt_ids, pad_id=0)     # [B,1,T,T]
    src_mask = make_src_mask(src_ids, pad_id=0)         # [B,1,1,S]

    print(f"   tgt_mask shape : {tgt_mask.shape}")      # [2, 1, 8, 8]
    print(f"   src_mask shape : {src_mask.shape}")      # [2, 1, 1, 10]

    print("\n   Decoder mask for batch item 0 (slice, row=query, col=key):")
    print("   1=attend, .=blocked (future OR padding)")
    tgt_labels = ["t0","t1","t2","t3","t4","t5","PAD","PAD"]
    visualize_attention_mask(tgt_mask[0, 0], tgt_labels)

    # ── Forward pass through one decoder layer ───────────────────────────
    print("[3] Forward pass through DecoderLayer...")
    enc_out      = torch.randn(B, SRC_LEN, D)
    tgt_emb      = torch.randn(B, TGT_LEN, D)
    decoder_layer = DecoderLayer(d_model=D, num_heads=8, d_ff=2048)
    decoder_layer.eval()

    with torch.no_grad():
        out = decoder_layer(tgt_emb, enc_out, tgt_mask=tgt_mask, src_mask=src_mask)

    print(f"   Output shape : {out.shape}")             # [2, 8, 512]

    # ── Inspect attention weights ─────────────────────────────────────────
    print("\n[4] Masked self-attention weights (head 0, batch item 0):")
    w = decoder_layer.masked_attn.attn_weights[0, 0]   # [T, T]
    print("   Upper triangle (future) max weight:",
          f"{w.triu(diagonal=1).max().item():.6f}  (should be ~0.0000)")
    print("   Diagonal (self) mean weight :",
          f"{w.diagonal().mean().item():.4f}")

    # ── Test with all-zero mask (blocked = -inf) ──────────────────────────
    print("\n[5] Attention score distribution check...")
    attn = ScaledDotProductAttention(dropout=0.0)
    Q = torch.randn(1, 1, 4, 64)
    K = torch.randn(1, 1, 4, 64)
    V = torch.randn(1, 1, 4, 64)
    mask = make_causal_mask(4)
    _, weights = attn(Q, K, V, mask)
    print("   Attention weights (head 0):")
    for i, row in enumerate(weights[0, 0]):
        vals = [f"{v:.3f}" for v in row.tolist()]
        print(f"   pos {i}: {vals}")
    print("   (upper-right values should be 0.000 — masked)")

    print("\n" + "=" * 66)
    print("  All checks passed!")
    print("=" * 66)


if __name__ == "__main__":
    run_demo()

  Masked Self-Attention — End-to-End Demo

[1] Causal mask for TGT_LEN=5  (1=allowed, .=blocked)
       BOS    I    love  cats  EOS  
------------------------------------
BOS     1     .     .     .     .   
I       1     1     .     .     .   
love    1     1     1     .     .   
cats    1     1     1     1     .   
EOS     1     1     1     1     1   

[2] Building decoder mask (causal + padding)...
   tgt_mask shape : torch.Size([2, 1, 8, 8])
   src_mask shape : torch.Size([2, 1, 1, 10])

   Decoder mask for batch item 0 (slice, row=query, col=key):
   1=attend, .=blocked (future OR padding)
       t0   t1   t2   t3   t4   t5  PAD  PAD 
---------------------------------------------
t0     1    .    .    .    .    .    .    .  
t1     1    1    .    .    .    .    .    .  
t2     1    1    1    .    .    .    .    .  
t3     1    1    1    1    .    .    .    .  
t4     1    1    1    1    1    .    .    .  
t5     1    1    1    1    1    1    .    .  
PAD    1    1    1    1    1  